# 1. Setup and Paths

In [ ]:
!unzip -q "/content/group_project.zip" -d "/content"

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from scipy import sparse
from scipy.sparse import load_npz
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/content/group_project")

P2_DIR = PROJECT_DIR / "person2_handoff" / "person2_split_preprocessing" / "outputs"
P3_DIR = PROJECT_DIR / "person3_handoff" / "outputs"
P4_DIR = PROJECT_DIR / "person4_handoff"

print("P2:", P2_DIR)
print("P3:", P3_DIR)
print("P4:", P4_DIR)

# 2. Load Data

In [ ]:
X_train = np.load(P2_DIR / "matrices" / "tree_structured_train.npy")
X_val = np.load(P2_DIR / "matrices" / "tree_structured_validation.npy")

train_labels = pd.read_csv(P2_DIR / "matrices" / "labels_train.csv")
val_labels = pd.read_csv(P2_DIR / "matrices" / "labels_validation.csv")

y_train = train_labels["aki_label"].to_numpy()
y_val = val_labels["aki_label"].to_numpy()

feature_names = pd.read_csv(P2_DIR / "preprocessors" / "tree_structured_feature_names.csv")["feature"].tolist()
split_manifest = pd.read_csv(P2_DIR / "split_manifest.csv")
baseline_metrics = pd.read_csv(P3_DIR / "validation_metrics.csv")

# 3. Verify Frozen Patient Split

In [ ]:
def verify_split(labels, partition):
    expected = (split_manifest[split_manifest["partition"] == partition]
        [["subject_id", "stay_id", "partition"]]
        .sort_values("stay_id").reset_index(drop=True))

    observed = (
        labels[["subject_id", "stay_id", "partition"]].sort_values("stay_id")
        .reset_index(drop=True))

    assert observed.equals(expected), f"{partition} does not match Person 2's frozen split"

verify_split(train_labels, "train")
verify_split(val_labels, "validation")

assert X_train.shape[0] == len(train_labels)
assert X_val.shape[0] == len(val_labels)

train_patients = set(train_labels["subject_id"])
val_patients = set(val_labels["subject_id"])

assert train_patients.isdisjoint(val_patients)

print("Train and validation encounters match Person 2's frozen split.")
print("No patient overlap between training and validation.")
print("Tree matrices align with label files.")

# 4. Check the Loaded Data

In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("Number of features:", len(feature_names))

print("Train AKI prevalence:", y_train.mean())
print("Validation AKI prevalence:", y_val.mean())

# 5. Load Baseline Model Results

In [ ]:
baseline_metrics

# 6. XGBoost Model Development

In [ ]:
def probability_metrics(y_true, y_prob):
    return {
        "average_precision": average_precision_score(y_true, y_prob),
        "auroc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob)}

def fit_xgboost(X_train, y_train, params):
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        **params)

    model.fit(X_train, y_train)
    return model

In [ ]:
base_params = {
    "n_estimators": 300,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8}

xgb_base = fit_xgboost(X_train, y_train, base_params)

In [ ]:
base_val_prob = xgb_base.predict_proba(X_val)[:, 1]
base_metrics = probability_metrics(y_val, base_val_prob)

pd.DataFrame([base_metrics])

# 7. Hyperparameter Tuning

In [ ]:
candidate_params = [
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 300, "max_depth": 2, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 400, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8},

    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 400, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8},

    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8}
]

In [ ]:
tuning_results = []

for i, params in enumerate(candidate_params, start=1):
    model = fit_xgboost(X_train, y_train, params)
    val_prob = model.predict_proba(X_val)[:, 1]
    metrics = probability_metrics(y_val, val_prob)

    tuning_results.append({"model_id": i,**params,**metrics})

tuning_results = pd.DataFrame(tuning_results)

In [ ]:
tuning_results = tuning_results.sort_values(
    "average_precision", ascending=False).reset_index(drop=True)

tuning_results

# 8. Select Best XGBoost Model

In [ ]:
best_row = tuning_results.iloc[0]

best_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "subsample": float(best_row["subsample"]),
    "colsample_bytree": float(best_row["colsample_bytree"])}

best_params

In [ ]:
best_xgb = fit_xgboost(X_train, y_train, best_params)

best_val_prob = best_xgb.predict_proba(X_val)[:, 1]
best_metrics = probability_metrics(y_val, best_val_prob)

pd.DataFrame([best_metrics])

# 9. Predictor Group Comparison

In [ ]:
age_gender_features = ["num__age", "cat__gender_M"]

age_gender_idx = [feature_names.index(f) for f in age_gender_features]

X_train_age_gender = X_train[:, age_gender_idx]
X_val_age_gender = X_val[:, age_gender_idx]

print("Age/gender train:", X_train_age_gender.shape)
print("Age/gender validation:", X_val_age_gender.shape)

In [ ]:
age_gender_xgb = fit_xgboost(X_train_age_gender,y_train,best_params)

age_gender_val_prob = age_gender_xgb.predict_proba(X_val_age_gender)[:, 1]

age_gender_metrics = probability_metrics(y_val,age_gender_val_prob)

pd.DataFrame([age_gender_metrics])

In [ ]:
# Load the original sparse text-inclusive matrices
X_train_text_raw = load_npz(P2_DIR / "matrices" / "tree_text_train.npz")
X_val_text_raw = load_npz(P2_DIR / "matrices" / "tree_text_validation.npz")

text_feature_names = pd.read_csv(P2_DIR / "preprocessors" / "tree_text_feature_names.csv")["feature"].tolist()

n_structured = len(feature_names)

# Confirm that the first 19 columns are the same structured features
assert text_feature_names[:n_structured] == feature_names

print("Raw train:", X_train_text_raw.shape)
print("Raw validation:", X_val_text_raw.shape)
print("Structured features:", n_structured)

In [ ]:
def dense_to_explicit_csr(X):
    X = np.asarray(X)

    n_rows, n_cols = X.shape
    data = X.ravel()

    indices = np.tile(np.arange(n_cols, dtype=np.int32),n_rows)

    indptr = np.arange(
        0,
        (n_rows + 1) * n_cols,
        n_cols,
        dtype=np.int64)

    return sparse.csr_matrix((data, indices, indptr),shape=X.shape)


# Preserve every structured value explicitly, including zero
X_train_structured_explicit = dense_to_explicit_csr(X_train)
X_val_structured_explicit = dense_to_explicit_csr(X_val)

# Keep only the chief-complaint/text part from the original sparse matrices
X_train_text_only = X_train_text_raw[:, n_structured:]
X_val_text_only = X_val_text_raw[:, n_structured:]

# Rebuild the text-inclusive matrices
X_train_text = sparse.hstack(
    [X_train_structured_explicit, X_train_text_only],
    format="csr")

X_val_text = sparse.hstack(
    [X_val_structured_explicit, X_val_text_only],
    format="csr")

print("Corrected train:", X_train_text.shape)
print("Corrected validation:", X_val_text.shape)

In [ ]:
train_structured_fixed = X_train_text[:, :n_structured]
val_structured_fixed = X_val_text[:, :n_structured]

print("Train zeros:")
print("Dense structured:", np.sum(X_train == 0))
print("Corrected sparse:", np.sum(train_structured_fixed.data == 0))

print("\nValidation zeros:")
print("Dense structured:", np.sum(X_val == 0))
print("Corrected sparse:", np.sum(val_structured_fixed.data == 0))

print("\nTrain NaNs:")
print("Dense structured:", np.isnan(X_train).sum())
print("Corrected sparse:", np.isnan(train_structured_fixed.data).sum())

print("\nValidation NaNs:")
print("Dense structured:", np.isnan(X_val).sum())
print("Corrected sparse:", np.isnan(val_structured_fixed.data).sum())

assert np.allclose(train_structured_fixed.toarray(),X_train,equal_nan=True)

assert np.allclose(val_structured_fixed.toarray(),X_val,equal_nan=True)

print("\nStructured values are preserved correctly.")

In [ ]:
text_xgb = fit_xgboost(X_train_text, y_train, best_params)

text_val_prob = text_xgb.predict_proba(X_val_text)[:, 1]
text_metrics = probability_metrics(y_val, text_val_prob)

pd.DataFrame([text_metrics])

In [ ]:
predictor_comparison = pd.DataFrame([
    {"predictor_group": "Age/Gender",
        "n_features": X_train_age_gender.shape[1],
        **age_gender_metrics},
    {"predictor_group": "Structured",
        "n_features": X_train.shape[1],
        **best_metrics},
    {"predictor_group": "Structured + Chief Complaint",
        "n_features": X_train_text.shape[1],
        **text_metrics}])

predictor_comparison

# 10. Tune Structured + Chief Complaint Model

In [ ]:
text_tuning_results = []

for i, params in enumerate(candidate_params, start=1):
    model = fit_xgboost(X_train_text, y_train, params)
    val_prob = model.predict_proba(X_val_text)[:, 1]
    metrics = probability_metrics(y_val, val_prob)

    text_tuning_results.append({"model_id": i,**params,**metrics})

text_tuning_results = pd.DataFrame(text_tuning_results)

text_tuning_results = text_tuning_results.sort_values(
    "average_precision", ascending=False).reset_index(drop=True)

text_tuning_results

In [ ]:
best_text_row = text_tuning_results.iloc[0]

best_text_params = {
    "n_estimators": int(best_text_row["n_estimators"]),
    "max_depth": int(best_text_row["max_depth"]),
    "learning_rate": float(best_text_row["learning_rate"]),
    "subsample": float(best_text_row["subsample"]),
    "colsample_bytree": float(best_text_row["colsample_bytree"])}

best_text_params

In [ ]:
best_text_xgb = fit_xgboost(X_train_text,y_train,best_text_params)

best_text_val_prob = best_text_xgb.predict_proba(X_val_text)[:, 1]

best_text_metrics = probability_metrics(y_val,best_text_val_prob)

pd.DataFrame([best_text_metrics])

#11. Final Model Comparison

In [ ]:
final_comparison = pd.concat([
    baseline_metrics[
        ["model_name", "predictive_features", "average_precision", "auroc", "brier_score"]],
    pd.DataFrame([
        {"model_name": "structured_xgboost",
            "predictive_features": X_train.shape[1],
            **best_metrics},
        {"model_name": "structured_chief_complaint_xgboost",
            "predictive_features": X_train_text.shape[1],
            **best_text_metrics}])], ignore_index=True)

final_comparison

# 12. Feature Interpretation

In [ ]:
importance = best_xgb.get_booster().get_score(importance_type="gain")

importance_df = pd.DataFrame([
    {"feature": feature_names[int(key[1:])],
        "importance": value}for key, value in importance.items()]).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.head(15)

In [ ]:
top_features = importance_df.head(15).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Gain Importance")
plt.ylabel("Feature")
plt.title("Top 15 Features in Structured XGBoost")
plt.tight_layout()
plt.show()

In [ ]:
text_feature_names = pd.read_csv(P2_DIR / "preprocessors" / "tree_text_feature_names.csv")["feature"].tolist()

print("Number of feature names:", len(text_feature_names))
print("Number of model features:", X_train_text.shape[1])

In [ ]:
text_importance = best_text_xgb.get_booster().get_score(importance_type="gain")

text_importance_df = pd.DataFrame([
    {"feature": text_feature_names[int(key[1:])],
        "importance": value}for key, value in text_importance.items()
]).sort_values("importance", ascending=False).reset_index(drop=True)

text_importance_df.head(20)

In [ ]:
top_text_features = text_importance_df.head(20).sort_values("importance")

plt.figure(figsize=(9, 7))
plt.barh(
    top_text_features["feature"],
    top_text_features["importance"])

plt.xlabel("Gain Importance")
plt.ylabel("Feature")
plt.title("Top 20 Features in Structured + Chief Complaint XGBoost")
plt.tight_layout()
plt.show()

# 13. Save Handoff Outputs

In [ ]:
import json
import joblib

OUTPUT_DIR = P4_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
tuning_results.to_csv(OUTPUT_DIR / "structured_tuning_record.csv", index=False)
text_tuning_results.to_csv(OUTPUT_DIR / "chief_complaint_tuning_record.csv", index=False)

predictor_comparison.to_csv(OUTPUT_DIR / "predictor_group_comparison.csv", index=False)
final_comparison.to_csv(OUTPUT_DIR / "final_model_comparison.csv", index=False)

importance_df.to_csv(OUTPUT_DIR / "structured_feature_importance.csv", index=False)
text_importance_df.to_csv(OUTPUT_DIR / "final_feature_importance.csv", index=False)

In [ ]:
validation_predictions = pd.concat([
    pd.DataFrame({
        "subject_id": val_labels["subject_id"],
        "stay_id": val_labels["stay_id"],
        "observed_label": y_val,
        "predicted_probability": age_gender_val_prob,
        "model_name": "age_gender_xgboost",
        "partition": "validation"
    }),
    pd.DataFrame({
        "subject_id": val_labels["subject_id"],
        "stay_id": val_labels["stay_id"],
        "observed_label": y_val,
        "predicted_probability": best_val_prob,
        "model_name": "structured_xgboost",
        "partition": "validation"
    }),
    pd.DataFrame({
        "subject_id": val_labels["subject_id"],
        "stay_id": val_labels["stay_id"],
        "observed_label": y_val,
        "predicted_probability": best_text_val_prob,
        "model_name": "structured_chief_complaint_xgboost",
        "partition": "validation"
    })], ignore_index=True)

validation_predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv",index=False)

validation_predictions.head()

In [ ]:
joblib.dump(age_gender_xgb,MODEL_DIR / "age_gender_xgboost.joblib")

joblib.dump(best_xgb,MODEL_DIR / "structured_xgboost.joblib")

joblib.dump(best_text_xgb,MODEL_DIR / "structured_chief_complaint_xgboost.joblib")

In [ ]:
model_settings = {
    "age_gender_xgboost": best_params,
    "structured_xgboost": best_params,
    "structured_chief_complaint_xgboost": best_text_params,
    "selection_metric": "average_precision",
    "test_data_used": False}

with open(OUTPUT_DIR / "model_settings.json", "w") as f:
    json.dump(model_settings, f, indent=2)

In [ ]:
top_text_features = text_importance_df.head(20).sort_values("importance")

plt.figure(figsize=(9, 7))
plt.barh(top_text_features["feature"], top_text_features["importance"])
plt.xlabel("Gain Importance")
plt.ylabel("Feature")
plt.title("Top 20 Features in Structured + Chief Complaint XGBoost")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "final_feature_importance.png",dpi=300,bbox_inches="tight")

In [ ]:
for path in sorted(P4_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(P4_DIR))

# 14. Reproducibility Checks

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import matplotlib
import joblib
import json

software_versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "matplotlib": matplotlib.__version__,
    "joblib": joblib.__version__}

with open(P4_DIR / "software_versions.json", "w") as f:
    json.dump(software_versions, f, indent=2)

In [ ]:
reloaded_age_gender_xgb = joblib.load(
    MODEL_DIR / "age_gender_xgboost.joblib")

reloaded_structured_xgb = joblib.load(
    MODEL_DIR / "structured_xgboost.joblib")

reloaded_text_xgb = joblib.load(
    MODEL_DIR / "structured_chief_complaint_xgboost.joblib")

reloaded_age_gender_prob = reloaded_age_gender_xgb.predict_proba(X_val_age_gender)[:, 1]

reloaded_structured_prob = reloaded_structured_xgb.predict_proba(X_val)[:, 1]

reloaded_text_prob = reloaded_text_xgb.predict_proba(X_val_text)[:, 1]

assert np.allclose(reloaded_age_gender_prob, age_gender_val_prob)
assert np.allclose(reloaded_structured_prob, best_val_prob)
assert np.allclose(reloaded_text_prob, best_text_val_prob)

print("All reloaded models reproduce the original validation predictions.")

In [ ]:
readme_text = f"""# Person 4 Handoff — XGBoost and Predictor Comparisons

## Objective

Develop and compare XGBoost models for predicting the primary AKI outcome using Person 2's frozen patient-level split and preprocessing outputs.

## Inputs

Person 4 reuses Person 2's split and preprocessing outputs.

Main inputs:

- `tree_structured_train.npy`
- `tree_structured_validation.npy`
- `tree_text_train.npz`
- `tree_text_validation.npz`
- `labels_train.csv`
- `labels_validation.csv`
- `split_manifest.csv`
- `tree_structured_feature_names.csv`
- `tree_text_feature_names.csv`

Person 3's validation metrics are also loaded for comparison with the baseline models.

## Data

Training set:

- {len(y_train):,} encounters
- AKI prevalence: {y_train.mean():.2%}

Validation set:

- {len(y_val):,} encounters
- AKI prevalence: {y_val.mean():.2%}

The frozen patient-level training and validation split from Person 2 was retained. Encounter assignments were checked against the split manifest and no patient overlap was found.

## Predictor Groups

Three XGBoost predictor groups were evaluated:

1. Age/Gender — {X_train_age_gender.shape[1]} features
2. Structured predictors — {X_train.shape[1]} features
3. Structured + Chief Complaint — {X_train_text.shape[1]:,} transformed features

Average Precision was used as the primary model-selection metric. AUROC and Brier score were reported as supporting metrics.

## Structured Zero Handling

Person 2's existing chief-complaint vocabulary and TF-IDF representation were retained.

The original text-inclusive matrix is sparse. Because absent sparse entries can be treated as missing by XGBoost, the structured feature block was reconstructed so that structured zero values were explicitly stored rather than treated as absent entries.

The corrected structured block was checked against the original dense structured matrix. Zero values and NaN values were preserved exactly in both the training and validation partitions.

The training and validation split was not changed.

## Structured XGBoost Tuning

A limited hyperparameter search was performed over:

- `n_estimators`
- `max_depth`
- `learning_rate`

`subsample` and `colsample_bytree` were fixed at 0.8.

Selected structured-model parameters:

- `n_estimators = {best_params["n_estimators"]}`
- `max_depth = {best_params["max_depth"]}`
- `learning_rate = {best_params["learning_rate"]}`
- `subsample = {best_params["subsample"]}`
- `colsample_bytree = {best_params["colsample_bytree"]}`

Validation performance:

- Average Precision: {best_metrics["average_precision"]:.6f}
- AUROC: {best_metrics["auroc"]:.6f}
- Brier score: {best_metrics["brier_score"]:.6f}

## Predictor Group Comparison

Using the fixed structured-XGBoost settings:

| Predictor group | Features | AP | AUROC | Brier |
|---|---:|---:|---:|---:|
| Age/Gender | {X_train_age_gender.shape[1]} | {age_gender_metrics["average_precision"]:.6f} | {age_gender_metrics["auroc"]:.6f} | {age_gender_metrics["brier_score"]:.6f} |
| Structured | {X_train.shape[1]} | {best_metrics["average_precision"]:.6f} | {best_metrics["auroc"]:.6f} | {best_metrics["brier_score"]:.6f} |
| Structured + Chief Complaint | {X_train_text.shape[1]} | {text_metrics["average_precision"]:.6f} | {text_metrics["auroc"]:.6f} | {text_metrics["brier_score"]:.6f} |

The structured + chief complaint model showed the strongest development performance and was therefore tuned further.

## Final Structured + Chief Complaint Model

Selected parameters:

- `n_estimators = {best_text_params["n_estimators"]}`
- `max_depth = {best_text_params["max_depth"]}`
- `learning_rate = {best_text_params["learning_rate"]}`
- `subsample = {best_text_params["subsample"]}`
- `colsample_bytree = {best_text_params["colsample_bytree"]}`

Validation performance:

- Average Precision: {best_text_metrics["average_precision"]:.6f}
- AUROC: {best_text_metrics["auroc"]:.6f}
- Brier score: {best_text_metrics["brier_score"]:.6f}

No test data were used during model development or hyperparameter selection.

## Feature Interpretation

Gain-based XGBoost feature importance was calculated for the final structured + chief complaint model.

Highly ranked chief-complaint features included:

- labs
- cardiac arrest
- renal failure
- abnormal labs
- fistula
- hypotension
- hyperkalemia

Structured predictors including systolic blood pressure, arrival transport, triage acuity, respiratory rate, diastolic blood pressure and gender also appeared among the top-ranked features.

Gain importance indicates how useful a feature was for prediction within the fitted model, but it does not show whether the feature increased or decreased predicted risk. Some chief-complaint terms may also reflect clinical information or concerns already present at ED arrival.

## Loading the Final Text-Inclusive Model

The final `structured_chief_complaint_xgboost` model must receive input with the same zero handling used during training.

Do not pass the original sparse `tree_text` matrix directly to the saved XGBoost model.

Before prediction:

1. Load the dense structured matrix.
2. Load the original sparse text-inclusive matrix.
3. Reconstruct the structured block so that zero values are explicitly stored.
4. Keep the original chief-complaint/text columns.
5. Combine the corrected structured block with the text block.
6. Pass the corrected matrix to the saved model.

This preprocessing must remain consistent for validation and future test evaluation.

## Outputs

`outputs/`

- `structured_tuning_record.csv`
- `chief_complaint_tuning_record.csv`
- `predictor_group_comparison.csv`
- `final_model_comparison.csv`
- `validation_predictions.csv`
- `final_feature_importance.csv`
- `model_settings.json`

`outputs/models/`

- `age_gender_xgboost.joblib`
- `structured_xgboost.joblib`
- `structured_chief_complaint_xgboost.joblib`

`outputs/figures/`

- `final_feature_importance.png`

## Final Model

The selected Person 4 model is:

`structured_chief_complaint_xgboost`

The model should remain frozen before independent held-out test evaluation.
"""

with open(P4_DIR / "README_P4.md", "w") as f:
    f.write(readme_text)

print(readme_text)
print("\nREADME_P4.md updated.")

In [ ]:
structured_lr_row = baseline_metrics[
    baseline_metrics["model_name"] == "structured_lr"
].iloc[0]

structured_text_lr_row = baseline_metrics[
    baseline_metrics["model_name"] == "structured_text_lr"
].iloc[0]

report_text = f"""# Person 4 XGBoost methods and validation results

## Methods contribution

XGBoost models were developed for the primary outcome of observed creatinine-defined AKI within 24 hours of ED arrival or earlier hospital discharge. All models used Person 2's fixed patient-separated training and validation partitions. The training set contained {len(y_train):,} encounters and the validation set contained {len(y_val):,} encounters.

Person 2's tree-model preprocessing was reused. Numeric missing values were retained as NaN so that XGBoost could handle missingness internally, while categorical variables were represented using the saved one-hot encoded features. For the structured-plus-chief-complaint model, Person 2's existing chief-complaint vocabulary and TF-IDF representation were retained. However, the structured feature block was reconstructed so that zero values were explicitly preserved in the sparse matrix rather than being treated as absent entries. The reconstructed structured values were checked against the original dense matrices, and both zero values and NaN values were preserved correctly.

Three predictor groups were compared on the same training and validation encounters: age and gender only, {X_train.shape[1]} structured features, and {X_train_text.shape[1]:,} transformed structured-plus-chief-complaint features. XGBoost models used a binary logistic objective and histogram-based tree construction. Average precision (AP) was used as the primary model-selection metric, with AUROC and Brier score reported as supporting metrics.

A limited hyperparameter search was performed using development data only. For the structured model, combinations of tree depth, learning rate and number of estimators were evaluated, while subsample and column subsampling were fixed at 0.8. The selected structured model used {best_params["n_estimators"]} estimators, maximum depth {best_params["max_depth"]} and learning rate {best_params["learning_rate"]}. The predictor groups were then compared using these fixed XGBoost settings.

Because the structured-plus-chief-complaint model showed the strongest validation performance, a further limited search was performed for this model. The selected configuration used {best_text_params["n_estimators"]} estimators, maximum depth {best_text_params["max_depth"]}, learning rate {best_text_params["learning_rate"]}, subsample {best_text_params["subsample"]} and column subsampling {best_text_params["colsample_bytree"]}. No test data were used during model development or hyperparameter selection.

## Validation results contribution

Using the fixed XGBoost settings, the age/gender predictor group achieved AP {age_gender_metrics["average_precision"]:.4f}, AUROC {age_gender_metrics["auroc"]:.4f} and Brier score {age_gender_metrics["brier_score"]:.4f}. The structured predictor model achieved AP {best_metrics["average_precision"]:.4f}, AUROC {best_metrics["auroc"]:.4f} and Brier score {best_metrics["brier_score"]:.4f}.

After correcting the sparse representation so that structured zero values were preserved consistently, the structured-plus-chief-complaint model achieved AP {text_metrics["average_precision"]:.4f}, AUROC {text_metrics["auroc"]:.4f} and Brier score {text_metrics["brier_score"]:.4f}.

After limited model-specific tuning, the final structured-plus-chief-complaint XGBoost model achieved AP {best_text_metrics["average_precision"]:.4f}, AUROC {best_text_metrics["auroc"]:.4f} and Brier score {best_text_metrics["brier_score"]:.4f} on the validation set.

For the structured-only models, structured XGBoost achieved AP {best_metrics["average_precision"]:.4f}, AUROC {best_metrics["auroc"]:.4f} and Brier score {best_metrics["brier_score"]:.4f}, compared with AP {structured_lr_row["average_precision"]:.4f}, AUROC {structured_lr_row["auroc"]:.4f} and Brier score {structured_lr_row["brier_score"]:.4f} for structured logistic regression.

For the text-inclusive models, the final XGBoost model achieved AP {best_text_metrics["average_precision"]:.4f}, AUROC {best_text_metrics["auroc"]:.4f} and Brier score {best_text_metrics["brier_score"]:.4f}, compared with AP {structured_text_lr_row["average_precision"]:.4f}, AUROC {structured_text_lr_row["auroc"]:.4f} and Brier score {structured_text_lr_row["brier_score"]:.4f} for text-inclusive logistic regression.

XGBoost therefore improved performance over logistic regression for both the structured-only and text-inclusive comparisons. Adding chief-complaint information produced a larger improvement in predictive performance for both modelling approaches.

## Interpretation and work remaining

The predictor-group comparison suggests that richer clinical information contributed substantially to predictive performance. Age and gender alone provided limited discrimination, while structured features improved both AP and AUROC. After keeping the representation of the structured predictors consistent, adding chief-complaint features further improved XGBoost performance.

Feature-importance analysis of the final model showed that many highly ranked features were derived from chief-complaint text. Terms such as labs, cardiac arrest, renal failure, abnormal labs, fistula, hypotension and hyperkalemia appeared among the highest-ranked features. Structured predictors including systolic blood pressure, arrival transport, triage acuity, respiratory rate, diastolic blood pressure and gender also appeared among the top features.

Gain-based feature importance shows how useful a feature was for prediction within the fitted model, but it does not show whether the feature increased or decreased the predicted risk. Some chief-complaint terms may also reflect clinical information or concerns already present when the patient arrived at the ED. These findings therefore indicate which features contributed to the model's predictions, but they do not show a causal relationship.

The final XGBoost configuration should now be frozen before independent test-set evaluation. Validation performance should still be treated as a development estimate because both predictor selection and hyperparameter selection used the validation partition.
"""

with open(P4_DIR / "P4_report_contribution.md", "w") as f:
    f.write(report_text)

print(report_text)
print("\nP4_report_contribution.md updated.")

In [ ]:
#%cd /content/group_project
#!zip -r /content/person4_handoff.zip person4_handoff